In [12]:
import sys 
import os
from pathlib import Path
import pandas as pd

#from word_embeddings import *
import torch

In [13]:
from csv import QUOTE_NONE
import sys
import csv

maxInt = sys.maxsize

while True:
    # decrease the maxInt value by factor 10 
    # as long as the OverflowError occurs.

    try:
        csv.field_size_limit(maxInt)
        break
    except OverflowError:
        maxInt = int(maxInt/10)


base_path = Path(os.path.abspath("")).parents[1] / "dataset_creation" / "data"
datasets = {
    "school_shooters": base_path / "school_shooters.csv",
    "manifestos": base_path / "manifestos.csv",
    "stair_twitter_archive": base_path / "stair_twitter_archive.csv",
    "twitter": base_path / "twitter.csv",
    "stream_of_consciousness": base_path / "stream_of_consciousness.csv"
}

schoolshootersinfo_df = pd.read_csv(datasets["school_shooters"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
#manifesto_df = pd.read_csv(datasets["manifestos"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stair_twitter_archive_df = pd.read_csv(datasets["stair_twitter_archive"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
twitter_df = pd.read_csv(datasets["twitter"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)
stream_of_consciousness_df = pd.read_csv(datasets["stream_of_consciousness"], encoding="utf-8", delimiter="‎", engine="python", quoting=QUOTE_NONE)

In [14]:
schoolshootersinfo_df["label"] = 1
#manifesto_df["shooter"] = 1
stair_twitter_archive_df["label"] = 1
twitter_df["label"] = 0
stream_of_consciousness_df["label"] = 0

In [15]:
from word_embeddings import preprocess_text, get_glove_word_vectors

In [16]:
shooter_df = pd.concat([schoolshootersinfo_df, stair_twitter_archive_df], ignore_index=True)[:100]
non_shooter_df = pd.concat([twitter_df[:100], stream_of_consciousness_df[:100]])
whole_corpus_df = pd.concat([shooter_df, non_shooter_df], ignore_index=True).sample(frac=1)

In [17]:
whole_corpus_df["text"] = whole_corpus_df["text"].map(lambda a: preprocess_text(a))

c:\Users\Jonas\NTNU-Masters\src\experiments\5_language_models\word_embeddings.py:69: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(text, "html.parser")


In [11]:
""" import bcolz
import numpy as np
import pickle

glove_path = Path(os.path.abspath("")).parents[2] / ".vector_cache/"

words = []
idx = 0
word2idx = {}
vectors = bcolz.carray(np.zeros(1), rootdir=f'{glove_path}/6B.50.dat', mode='w')

with open(f'{glove_path}/glove.6B.50d.txt', 'rb') as f:
    for l in f:
        line = l.decode().split()
        word = line[0]
        words.append(word)
        word2idx[word] = idx
        idx += 1
        vect = np.array(line[1:]).astype(float)
        vectors.append(vect)
    
vectors = bcolz.carray(vectors[1:].reshape((400000, 50)), rootdir=f'{glove_path}/6B.50.dat', mode='w')
vectors.flush()
pickle.dump(words, open(f'{glove_path}/6B.50_words.pkl', 'wb'))
pickle.dump(word2idx, open(f'{glove_path}/6B.50_idx.pkl', 'wb')) """

In [18]:
vectors = bcolz.open(f'{glove_path}/6B.50.dat')[:]
words = pickle.load(open(f'{glove_path}/6B.50_words.pkl', 'rb'))
word2idx = pickle.load(open(f'{glove_path}/6B.50_idx.pkl', 'rb'))

glove = {w: vectors[word2idx[w]] for w in words}

In [31]:
vocab = {}

i = 0
for t in whole_corpus_df["text"]:
    for word in t:
        if word not in vocab:
            vocab[word] = i
            i += 1

vocab_len = len(vocab)
print(vocab_len)

6186


In [32]:
emb_dim = 50 # This is out_dim for embedding layer

weights_matrix = np.zeros((vocab_len, emb_dim))
words_found = 0

for i, word in enumerate(vocab):
    try:
        weights_matrix[i] = glove[word]
        words_found += 1
    except KeyError:
        np.random.normal(scale=0.6, size=(emb_dim, ))

In [33]:
print(weights_matrix[0])

[ 0.88387   -0.14199    0.13566    0.098682   0.51218    0.49138
 -0.47155   -0.30742    0.01963    0.12686    0.073524   0.35836
 -0.60874   -0.18676    0.78935    0.54534    0.1106    -0.2923
  0.059041  -0.69551   -0.18804    0.19455    0.32269   -0.49981
  0.306     -2.3902    -0.60749    0.37107    0.078912  -0.23896
  3.839     -0.20355   -0.35613   -0.69185   -0.17497   -0.35323
  0.10598   -0.039303   0.015701   0.038279  -0.35283    0.44882
 -0.16534    0.31579    0.14963   -0.071277  -0.53506    0.52711
 -0.20148    0.0095952]


In [34]:
from sklearn.model_selection import train_test_split

In [ ]:
# Setting up params for neural net

longest_sequence = max([len(t) for t in whole_corpus_df["text"]]) # input_length

input_len = longest_sequence
input_dim = vocab_len + 1 # +1 to handle out of vocab words
out_dim = emb_dim



In [35]:
x_train, x_test, y_train, y_test = train_test_split(whole_corpus_df["text"], whole_corpus_df["shooter"], test_size=0.1)
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1)

In [36]:
# Check sizes of tensors to shape input to mlp model
shapes = [list(t.shape) for t in whole_corpus_df["text"]]
dic_size = [dims[0] for dims in shapes]
print(shapes)


max_dic_size = max(dic_size)
min_dic_size = min(dic_size)
print(min_dic_size)
print(max_dic_size)
for x in x_train:
    if x.shape[0] == min_dic_size:
        print("x_shape: ", x.shape)
        i = 0
        for _ in x:
            print(_)
            i += 1
            print("i: ", i)

[[21, 50], [16, 50], [255, 50], [14, 50], [14, 50], [18, 50], [269, 50], [370, 50], [13, 50], [10, 50], [22, 50], [389, 50], [33, 50], [210, 50], [247, 50], [152, 50], [260, 50], [19, 50], [20, 50], [636, 50], [7, 50], [325, 50], [464, 50], [339, 50], [322, 50], [243, 50], [18, 50], [238, 50], [327, 50], [4, 50], [10, 50], [375, 50], [12, 50], [9, 50], [9, 50], [7, 50], [284, 50], [179, 50], [221, 50], [76, 50], [1, 50], [6, 50], [473, 50], [13, 50], [6, 50], [168, 50], [11, 50], [14, 50], [11, 50], [14, 50], [12, 50], [12, 50], [12, 50], [10, 50], [4, 50], [302, 50], [452, 50], [10, 50], [9, 50], [273, 50], [343, 50], [15, 50], [13, 50], [17, 50], [6, 50], [20, 50], [8, 50], [661, 50], [239, 50], [352, 50], [42, 50], [3, 50], [39, 50], [10, 50], [11, 50], [4, 50], [4, 50], [11, 50], [4, 50], [2, 50], [8, 50], [9, 50], [13, 50], [9, 50], [30, 50], [9, 50], [6, 50], [42, 50], [10, 50], [321, 50], [7, 50], [12, 50], [266, 50], [2, 50], [15, 50], [13, 50], [14, 50], [530, 50], [14, 50], [